# 📊 Task 3: Exploratory Data Analysis (EDA)
## Edutech Solution – AI & ML Internship

**Dataset:** Iris (150 rows × 5 columns — 3 species, 4 features)  
**Tools:** Python · Pandas · Matplotlib · Seaborn  
**Objective:** Extract meaningful stories from raw data through visualizations and statistics

---

## 📦 Step 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="Set2")
COLORS = ["#2ecc71", "#3498db", "#e74c3c"]
print("✅ Libraries imported!")

## 📂 Step 1 — Load Dataset & Overview

In [ ]:
df = pd.read_csv("iris.csv")
print(f"Shape: {df.shape}")
print(f"Species: {df['species'].unique().tolist()}")
print(f"Class counts:\n{df['species'].value_counts()}")
df.head(10)

In [ ]:
print("Summary Statistics:")
df.describe().round(3)

In [ ]:
print("Per-species mean values:")
num_cols = ["sepal_length","sepal_width","petal_length","petal_width"]
df.groupby("species")[num_cols].mean().round(2)

## 📈 Step 2 — Univariate Analysis
> Examining each variable **independently** using histograms and boxplots.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle("Univariate Analysis — Histograms & Boxplots", fontsize=16, fontweight='bold')

for i, col in enumerate(num_cols):
    ax_h = axes[0][i]
    for j, (sp, grp) in enumerate(df.groupby("species")):
        ax_h.hist(grp[col], bins=15, alpha=0.65, color=COLORS[j], label=sp, edgecolor='white')
    ax_h.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.2)
    ax_h.set_title(f"{col.replace('_',' ').title()}", fontweight='bold')
    ax_h.set_xlabel(col.replace('_',' '))
    ax_h.legend(fontsize=8)

    ax_b = axes[1][i]
    data = [df[df["species"]==sp][col].values for sp in df["species"].unique()]
    bp = ax_b.boxplot(data, labels=df["species"].unique(), patch_artist=True,
                      medianprops=dict(color="black", linewidth=2))
    for patch, c in zip(bp['boxes'], COLORS):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax_b.set_xticklabels(df["species"].unique(), rotation=10)

plt.tight_layout()
plt.savefig("01_univariate_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

## 🔗 Step 3 — Bivariate Analysis
> Examining **relationships between two variables** using scatter plots and bar charts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Bivariate Analysis — Scatter Plots", fontsize=14, fontweight='bold')

pairs = [("sepal_length","sepal_width"), ("petal_length","petal_width"), ("sepal_length","petal_length")]
for ax, (x, y) in zip(axes, pairs):
    for j, (sp, grp) in enumerate(df.groupby("species")):
        ax.scatter(grp[x], grp[y], label=sp, color=COLORS[j], alpha=0.75, s=60, edgecolors='white')
    ax.set_xlabel(x.replace('_',' ').title()); ax.set_ylabel(y.replace('_',' ').title())
    ax.set_title(f"{x.replace('_',' ').title()} vs\n{y.replace('_',' ').title()}", fontweight='bold')
    ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("02_bivariate_scatter.png", dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("Mean Feature Values per Species", fontsize=14, fontweight='bold')
means = df.groupby("species")[num_cols].mean()
for ax, col in zip(axes, num_cols):
    bars = ax.bar(means.index, means[col], color=COLORS, edgecolor='white', width=0.5)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.02,
                f'{bar.get_height():.2f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(col.replace('_',' ').title(), fontweight='bold')
    ax.set_xticklabels(means.index, rotation=10)
plt.tight_layout(); plt.savefig("03_bivariate_bar.png", dpi=150, bbox_inches='tight'); plt.show()

## 🌡️ Step 4 — Correlation Matrix & Heatmap
> Values close to **+1** = strong positive correlation; close to **-1** = strong negative.

In [ ]:
corr = df[num_cols].corr()
print("Correlation Matrix:")
corr.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Correlation Heatmaps", fontsize=14, fontweight='bold')

sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.5, ax=axes[0], square=True, cbar_kws={"shrink":0.8})
axes[0].set_title("Full Heatmap", fontweight='bold')

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=axes[1], square=True, cbar_kws={"shrink":0.8})
axes[1].set_title("Lower Triangle Heatmap", fontweight='bold')

plt.tight_layout(); plt.savefig("04_correlation_heatmap.png", dpi=150, bbox_inches='tight'); plt.show()
print("\nStrongest correlation: petal_length ↔ petal_width =", round(corr['petal_length']['petal_width'],3))

## 🔷 Step 5 — Pairplot
> Visualizes **all pairwise relationships** at once with KDE on the diagonal.

In [ ]:
g = sns.pairplot(df, hue="species",
                 palette={"setosa":COLORS[0],"versicolor":COLORS[1],"virginica":COLORS[2]},
                 diag_kind="kde", plot_kws={"alpha":0.65,"s":40,"edgecolor":"white"},
                 height=2.5)
g.fig.suptitle("Pairplot — Iris Dataset", y=1.02, fontsize=14, fontweight='bold')
g.savefig("05_pairplot.png", dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Step 6 — Patterns, Trends & Anomalies

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Patterns, Trends & Anomalies", fontsize=15, fontweight='bold')

# CDF
for j,(sp,grp) in enumerate(df.groupby("species")):
    axes[0][0].plot(sorted(grp["petal_length"]), np.linspace(0,1,len(grp)),
                    color=COLORS[j], label=sp, linewidth=2)
axes[0][0].set_title("Petal Length — CDF", fontweight='bold')
axes[0][0].set_xlabel("Petal Length (cm)"); axes[0][0].set_ylabel("Cumulative Probability")
axes[0][0].legend()

# Violin
species_list = list(df["species"].unique())
data_v = [df[df["species"]==sp]["petal_length"].values for sp in species_list]
vp = axes[0][1].violinplot(data_v, positions=[1,2,3], showmedians=True)
for i,pc in enumerate(vp['bodies']): pc.set_facecolor(COLORS[i]); pc.set_alpha(0.7)
axes[0][1].set_xticks([1,2,3]); axes[0][1].set_xticklabels(species_list)
axes[0][1].set_title("Petal Length — Violin Plot", fontweight='bold')

# Pie
counts = df["species"].value_counts()
axes[1][0].pie(counts, labels=counts.index, colors=COLORS, autopct='%1.0f%%',
               startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1][0].set_title("Species Distribution", fontweight='bold')

# Area scatter
df["sepal_area"] = df["sepal_length"] * df["sepal_width"]
df["petal_area"] = df["petal_length"] * df["petal_width"]
for j,(sp,grp) in enumerate(df.groupby("species")):
    axes[1][1].scatter(grp["sepal_area"], grp["petal_area"],
                       label=sp, color=COLORS[j], alpha=0.7, s=60, edgecolors='white')
axes[1][1].set_xlabel("Sepal Area (cm²)"); axes[1][1].set_ylabel("Petal Area (cm²)")
axes[1][1].set_title("Sepal Area vs Petal Area", fontweight='bold'); axes[1][1].legend()

plt.tight_layout(); plt.savefig("06_patterns_trends.png", dpi=150, bbox_inches='tight'); plt.show()

## 📦 Step 7 — Outlier Detection
> Red dots in boxplots indicate **outliers** (beyond 1.5×IQR).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle("Outlier Detection — Boxplots per Feature & Species", fontsize=14, fontweight='bold')

for ax, col in zip(axes, num_cols):
    data = [df[df["species"]==sp][col].values for sp in species_list]
    bp = ax.boxplot(data, patch_artist=True, labels=species_list,
                    medianprops=dict(color="black", linewidth=2),
                    flierprops=dict(marker='o',markersize=7,markerfacecolor='red',markeredgecolor='black'))
    for patch,c in zip(bp['boxes'],COLORS): patch.set_facecolor(c); patch.set_alpha(0.7)
    ax.set_title(col.replace('_',' ').title(), fontweight='bold')
    ax.set_xticklabels(species_list, rotation=12)

plt.tight_layout(); plt.savefig("07_outlier_detection.png", dpi=150, bbox_inches='tight'); plt.show()

## ✅ Step 8 — Key Insights Summary

| Insight | Finding |
|---------|---------|
| Best separating feature | Petal Length & Petal Width |
| Strongest correlation | petal_length ↔ petal_width = **0.963** |
| Most distinct species | Setosa (clearly separated) |
| Overlap | Versicolor & Virginica (sepal dims) |
| Outliers | Setosa sepal_width has a few outliers |
| Missing values | None — dataset is ML-ready |

### Interview Q&A

**Q1. Univariate vs Bivariate vs Multivariate?**  
Univariate = 1 variable (histograms, boxplots). Bivariate = 2 variables (scatter, bar). Multivariate = 3+ variables (pairplot, heatmap).

**Q2. How to interpret a Correlation Heatmap?**  
Values near +1 = strong positive, near -1 = strong negative, near 0 = no linear relationship. Color intensity shows strength.

**Q3. Purpose of a Boxplot?**  
Shows median, quartiles, spread, and outliers (dots beyond whiskers) in one compact chart.

**Q4. How to identify outliers visually?**  
Boxplot dots beyond whiskers; scatter plot isolated points; histogram long tails.

**Q5. Why is EDA the most important step before modeling?**  
EDA reveals data quality issues, feature distributions, correlations, and patterns — preventing garbage-in-garbage-out and guiding feature engineering.